<div style="font-family: Arial, sans-serif; line-height: 1.6; color: #fab956; max-width: 1200px; margin: 0 auto; padding: 20px; border: 1px solid #000000; border-radius: 8px; background-color: #00183a74;">

  <h1 style="color: #c25719; border-bottom: 2px solid #c25719; padding-bottom: 8px; margin-top: 0;">
    Predicting Lung Nodule Malignancy from CT Data
  </h1>

  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
   What data do we have?
  </h3>

  <p>
  
| Dataset | What it contains | How we use it |
|---|---|---|
| LNDb train | 236 CT scans, with 1219 unique findings (768 nodules), segmentation masks for nodules >= 3 mm | Main training data |
| LNDb test | 58 CT scans and 2900 candidates (50 per CT) with only a centroid | Inference only. Every candidate is predicted as if it were a nodule |
| LIDC IDRI | 1010 patients, 1018 CT series, up to four readings per nodule | Annotations through `pylidc`; patches cut from the DICOM files; used to train the size model |
  </p>
  <p>
  Malignancy is evaluated in LNDb's file <strong>'chars_trainNodules.csv'</strong>.
  </p>
  
  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
    Which files to consider? And which to discard?
  </h3>

  <ol style="padding-left: 20px; margin-bottom: 20px;">
    <li style="margin-bottom: 8px;">Kept:
      <ul style="padding-left: 20px; margin-bottom: 20px;">
      <li style="margin-bottom: 8px;">The merged findings file <strong>'trainNodules_gt.csv'</strong>. The unit of analysis is the nodule, not the reading, because one nodule can be read by up to three radiologists and counting it three times would leak information which is a crucial error we made sure not to make.</li>
      <li style="margin-bottom: 8px;"><strong>'chars_trainNodules.csv'</strong> for the malignancy ratings, <strong>'LNDbAcqParams.csv'</strong> for scanner information, and the masks for nodules of 3 mm or more.</li>
      <li style="margin-bottom: 8px;">The official LNDb folds <strong>'trainFolds.csv'</strong>, split by CT, used to confirm the final choice on splits that played no part in it.</li>
      </ul>
    </li>
    <li style="margin-bottom: 8px;">Dropped:
      <ul style="padding-left: 20px; margin-bottom: 20px;">
        <li style="margin-bottom: 8px;">The per patient diagnosis file of LIDC. It says whether a patient has cancer but not which nodule is the malignant one, so using it as a label would give every nodule of that patient the same wrong answer. Therefore, we did not consider it useful for our main goal.</li>
        <li style="margin-bottom: 8px;">Fleischner files and medical reports. Fleischner is a follow up recommendation, not malignancy, and reports do not exist for the test scans.</li>
        <li style="margin-bottom: 8px;">The original <strong>'.rar'</strong> archives, the X ray series of LIDC (DX and CR) and the macOS <strong>'._*'</strong> files.</li>
        </ul>
    </li>
    <li style="margin-bottom: 8px;">Practical Notes:</li>
      <ul style="padding-left: 20px; margin-bottom: 20px;">
        <li style="margin-bottom: 8px;">In the LNDb files, Malignancy is averaged in <strong>'allNods.csv'</strong>.</li>
      </ul>
    </li>
  </ol>

  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
   Key Findings in the Literature
  </h3>

  <p>
  <strong> Size alone explains a lot -</strong> Baltatzis et al. (2021) show that a baseline that only uses nodule diameter reaches 83% to 85% accuracy on LIDC, and that the way labels are aggregated can matter more than the model. Mean and median aggregation give 1010 and 848 usable nodules respectively. We will always report a diameter only baseline and use that for any comparision needed.
  </p>
  <p>
  <strong>Splits must respect patients -</strong> Causey et al. criticise works that mix nodules of the same patient across training and validation. Nodules from one scan share scanner, protocol and anatomy, so we we decided to always validate by CT scan.
  </p>
  <p>
  <strong>Tools -</strong> PyRadiomics (van Griethuysen et al., 2017) is our feature extractor and the IBSI work (Zwanenburg et al., 2020) is the reference for feature definitions. We use a fixed bin width and resample to 1 mm isotropic voxels.
  </p>
  <p>
  <strong>Domain shift -</strong> LNDb comes from one hospital with three Siemens scanner models, while LIDC is multi centre and multi vendor. LNDb was designed as an external set for models trained on LIDC. We first rely on resampling and normalisation, and only consider ComBat harmonisation (Orlhac et al., 2022) if the features show a clear scanner effect.
  </p>
  <p>
  <strong>Pretrained features -</strong> The foundation model of Pai et al. (2024) only needs the lesion centroid, exactly what our test file has. It is an option for later, since our team decided not to use deep learning for now.
  </p>

  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
   Decisions
  </h3>

  | Decision | Choice | Why |
|----|----|----|
| Unit of analysis | Unique nodule | Avoids counting the same nodule several times |
| Training rows | Nodules with malignancy rating only | Non-nodules have no rating and only add noise |
| Training label | Mean of the non zero ratings, rounded half up | Keeps the 1 to 5 scale |
| Region of interest | Same procedure from the centroid in training and test | The test set has only coordinates. Masks serve only to check our segmentation and robustness of features |
| Features | 21 handcrafted features and 9 sphere features; PyRadiomics analysed (IBSI configuration, ICC) but not used because it did not help | Simple, explainable, robust |
| Size | Learned from the radiologists' diameters, which exist only in training | Our segmentation measures size poorly; learning using privileged information (Vapnik and Vashist, 2009) |
| Model | Ordinal logistic regression with an L2 penalty | Ordered classes, few parameters (McCullagh, 1980) |
| Decision rule | Most probable class after a partial correction for class frequencies (power 0.5 for the rarest classes) | Balances the four competition metrics (Saerens et al., 2002) |
| LIDC | Trains the size model only | Physical size is comparable between datasets; the rating is not thanks to the different context of the datasets. In the classifier LIDC made results worse |
| Validation | Grouped by CT/patient, stratified, repeated; intervals by resampling CTs; confirmation on new seeds and on the official folds | Leakage control (Kapoor and Narayanan, 2023), small sample error bars (Varoquaux, 2018), selection bias (Cawley and Talbot, 2010) |
| Metrics | The four competition metrics plus a composite (BA + F1 + QWK - MAE), within one class, sensitivity for ratings 4 and 5 | The ranking is a mean rank; the extra metrics are for our decisions and the report |

# Part 2. All the code of the project



In [ ]:
import warnings
warnings.filterwarnings("ignore")

## 2.1 Patches

### Cutting 3D patches (LNDb)


Cuts 3D patches around nodule centres, with the same procedure for training and test.

A patch is a cube of 64 mm around a centre given in world coordinates (mm). It is resampled to 1 mm
voxels, so the array has 64 x 64 x 64 voxels and the centre falls exactly on voxel (32, 32, 32).
The values are raw Hounsfield units (int16). The array axes are (z, y, x) and they are aligned with the
world axes, whatever the tilt of the CT, so every patch is oriented in the same way. Every feature is then measured in three dimensions inside this cube: the segmentation, size, shape, density, the ring around the nodule, the sphere features and the radiomic features.


### Why 3D Patches

- A nodule and a vessel can look the same in one slice. A vessel cut across looks like a small round blob on an axial slice, exactly like a nodule. Only along the third axis does it reveal itself as a tube. Our segmentation relies on this: it removes the parts of the region that extend away from the nodule, which are mostly vessels and pleura. The detection literature shows the same effect. Setio et al. (2016) found that combining several views of a candidate works better than a single view, and Dou et al. (2017) showed that 3D context helps to tell nodules from false positives. Although thee studies use neural networks, the point still holds for any method since the third dimension carries information that a slice does not.

- Size is the strongest predictor, and it is best measured as a volume. The Fleischner Society guidelines give their size thresholds both as diameters and as volumes (MacMahon et al., 2017). Volume, sphericity, elongation and flatness only exist in 3D, and a diameter measured on one slice depends on which slice is chosen.

- Radiomic features are defined in 3D. The Image Biomarker Standardisation Initiative defines its shape and texture features on volumes and recommends isotropic voxels before computing them (Zwanenburg et al., 2020). Texture features change when the voxel size changes (Shafiq-ul-Hassan et al., 2017). Resampling every patch to 1 mm in all directions makes the features comparable between scans with different pixel sizes and slice spacings.

- The data support it. Almost all LNDb scans have 1 mm slices, so a 1 mm cube adds very little interpolation. The test file gives each candidate as a point in world coordinates (x, y, z), so a cube around that point is the natural unit. The cube is also what the LNDb authors distribute (the script getNoduleCubes.py cuts cubes of 51 mm). Recent work does the same: the foundation model of Pai et al. (2024) takes 50 mm cubes at 1 mm around the lesion centre.

### Why 64 mm?

- The largest rated LNDb nodule measures about 31 mm. A 64 mm cube holds it with room for the surrounding tissue (vessels, pleura, chest wall), which the ring features and the vessel removal need. Our choice for the specific number of 64 mm is a practical choice, close to the 50 to 51 mm used by the references.


Dps adicionar no final
### References
Dou, Q., Chen, H., Yu, L., Qin, J. and Heng, P.-A. (2017). Multilevel contextual 3-D CNNs for false positive reduction in pulmonary nodule detection. IEEE Transactions on Biomedical Engineering, 64(7), 1558 to 1567.
MacMahon, H. et al. (2017). Guidelines for management of incidental pulmonary nodules detected on CT images: from the Fleischner Society 2017. Radiology, 284(1), 228 to 243.
Pai, S. et al. (2024). Foundation model for cancer imaging biomarkers. Nature Machine Intelligence.
Setio, A. A. A. et al. (2016). Pulmonary nodule detection in CT images: false positive reduction using multi-view convolutional networks. IEEE Transactions on Medical Imaging, 35(5), 1160 to 1169.
Shafiq-ul-Hassan, M. et al. (2017). Intrinsic dependencies of CT radiomic features on voxel size and number of gray levels. Medical Physics, 44(3), 1050 to 1062.
Zwanenburg, A. et al. (2020). The Image Biomarker Standardization Initiative: standardized quantitative radiomics for high-throughput image-based phenotyping. Radiology, 295(2), 328 to 338.

In [ ]:
import argparse
import os
import resource
import time
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk

PATCH_SIZE = 64           # voxels per side
SPACING_MM = 1.0          # voxel size of the patch
CENTRE_INDEX = PATCH_SIZE // 2   # voxel that holds the centre: 32
AIR_HU = -1024            # value used outside the CT, the floor of the usual HU normalisation

REPO_ROOT = Path.cwd().parent          
DATA_DIR = Path(os.environ.get("MASS_DATA_DIR", REPO_ROOT / "data"))
LNDB_DIR = DATA_DIR / "LNDb"
PATCHES_DIR = DATA_DIR / "patches"


def find_ct_files():
    #Maps each LNDb CT id to its .mhd file, in the training and test folders
    files = {}
    for pattern in ("data/data[0-5]/LNDb-*.mhd", "data/testdata[01]/LNDb-*.mhd"):
        for path in LNDB_DIR.glob(pattern):
            files[int(path.stem.split("-")[1])] = path
    return files


def load_findings(split):
    #Findings of the split with their centre in world coordinates (mm)
    if split == "train":
        table = pd.read_csv(LNDB_DIR / "LNDb_csvs" / "trainNodules_gt.csv")
    else:
        table = pd.read_csv(LNDB_DIR / "testNodules.csv")
    return table[["LNDbID", "FindingID", "x", "y", "z"]]


def centre_is_inside(image, centre_mm):
    # Gives True when the physical point falls inside the volume of the image
    index = np.array(image.TransformPhysicalPointToContinuousIndex(tuple(float(v) for v in centre_mm)))
    size = np.array(image.GetSize())
    return bool(np.all(index >= -0.5) and np.all(index <= size - 0.5))


def extract_patch(image, centre_mm, interpolator=sitk.sitkLinear, outside_value=AIR_HU):
    """Cuts a 64 mm cube around a world point and returns (patch, outside_fraction).

    The cube is described directly in physical space: 1 mm voxels, axes aligned with the world axes
    and the origin chosen so that the centre falls on voxel (32, 32, 32). SimpleITK then looks up each
    voxel in the CT using the origin, spacing and direction of the CT, so a tilted CT is handled correctly.
    Points outside the CT get outside_value, and outside_fraction says how many voxels they are.
    """
    origin = np.asarray(centre_mm, dtype=float) - CENTRE_INDEX * SPACING_MM

    resampler = sitk.ResampleImageFilter()
    resampler.SetSize([PATCH_SIZE] * 3)
    resampler.SetOutputSpacing([SPACING_MM] * 3)
    resampler.SetOutputDirection(tuple(np.eye(3).ravel()))
    resampler.SetOutputOrigin(tuple(origin))
    resampler.SetInterpolator(interpolator)
    resampler.SetOutputPixelType(sitk.sitkFloat32)
    resampler.SetDefaultPixelValue(float("nan"))      # marks the voxels that fall outside the CT

    patch = sitk.GetArrayFromImage(resampler.Execute(image))   # numpy order is (z, y, x)
    outside = np.isnan(patch)
    patch[outside] = outside_value
    return np.rint(patch).astype(np.int16), float(outside.mean())


def cut_scan_patches(ct_id, ct_path, findings, split, out_dir):
    #Cuts and saves the patches of every finding of one CT. Returns one row per finding
    image = sitk.ReadImage(str(ct_path))
    rows = []
    for finding in findings.itertuples():
        patch_id = f"LNDb-{ct_id:04d}_f{finding.FindingID:03d}"
        centre = (finding.x, finding.y, finding.z)
        row = {"patch_id": patch_id, "split": split, "ct_id": ct_id, "finding_id": finding.FindingID,
               "x_mm": finding.x, "y_mm": finding.y, "z_mm": finding.z,
               "status": "ok", "outside_fraction": np.nan, "hu_min": np.nan, "hu_max": np.nan, "file": ""}
        try:
            index = image.TransformPhysicalPointToContinuousIndex(tuple(float(v) for v in centre))
            row.update(ct_index_x=index[0], ct_index_y=index[1], ct_index_z=index[2])
            if not centre_is_inside(image, centre):
                row["status"] = "centre_outside_volume"
            else:
                patch, outside_fraction = extract_patch(image, centre)
                relative_path = f"{split}/{patch_id}.npy"
                np.save(out_dir / relative_path, patch)
                row.update(outside_fraction=outside_fraction, hu_min=int(patch.min()), hu_max=int(patch.max()),
                           file=relative_path)
        except Exception as error:              
            row["status"] = f"error: {error}"
        rows.append(row)
    return rows


def cut_lndb_patches(split, limit_cts=None, overwrite=False, out_dir=PATCHES_DIR):
    # Processes one CT at a time (to keep the memory low) and writes one file per CT
    ct_files = find_ct_files()
    findings = load_findings(split)
    (out_dir / split).mkdir(parents=True, exist_ok=True)
    (out_dir / "manifests").mkdir(exist_ok=True)

    ct_ids = sorted(findings["LNDbID"].unique())[:limit_cts]
    seconds = []
    for ct_id in ct_ids:
        manifest_file = out_dir / "manifests" / f"{split}_LNDb-{ct_id:04d}.csv"
        if manifest_file.exists() and not overwrite:
            continue                              # already done, so a long run can be resumed
        started = time.time()
        rows = cut_scan_patches(ct_id, ct_files[ct_id], findings[findings["LNDbID"] == ct_id], split, out_dir)
        pd.DataFrame(rows).to_csv(manifest_file, index=False)
        seconds.append(time.time() - started)
        print(f"{split} CT {ct_id}: {len(rows)} findings in {seconds[-1]:.1f} s")

    manifests = sorted((out_dir / "manifests").glob(f"{split}_LNDb-*.csv"))
    manifest = pd.concat([pd.read_csv(f) for f in manifests], ignore_index=True)
    manifest.to_csv(out_dir / f"manifest_{split}.csv", index=False)
    return manifest, seconds


def summarise_manifest(manifest, split, expected_rows=None):
    #Prints the checks that matter after a run
    print(f"\n=== {split}: {len(manifest)} rows in the manifest" +
          (f" (expected {expected_rows})" if expected_rows else ""))
    print("status:", manifest["status"].str.split(":").str[0].value_counts().to_dict())
    ok = manifest[manifest["status"] == "ok"]
    for limit in (0.05, 0.25, 0.5):
        print(f"outside_fraction above {limit}: {(ok['outside_fraction'] > limit).sum()}")
    duplicated = manifest.round(3).duplicated(["ct_id", "x_mm", "y_mm", "z_mm"], keep=False)
    print(f"rows that share the same CT and position with another row: {int(duplicated.sum())}")
    if len(ok):
        print(f"HU range of the patches: min {ok['hu_min'].min()}, max {ok['hu_max'].max()}")

### The same patches for LIDC-IDRI


Cuts the same 64 mm patches around the LIDC-IDRI nodules, so LIDC can be used as extra training data.

#### Why a separate script?
The LNDb scans are .mhd files, the LIDC scans are folders of DICOM slices. Everything after reading the scan is shared with LNDb (the function extract_patch in patches.py), so a LIDC patch and a LNDb patch are cut in exactly the same way: 1 mm voxels, world axes, raw Hounsfield units.

#### How the scan is read?
SimpleITK reads the DICOM series, sorts the slices along the scan axis and applies RescaleSlope and RescaleIntercept, so the voxels are already in Hounsfield units.

#### Where the nodule is?
The csv nodule_table.csv keeps the centre of every LIDC nodule as voxel indices (row, column, slice) taken from pylidc. SimpleITK turns these indices into world coordinates using the origin, spacing and
direction of the scan, which also works for the few scans whose slices are not axis aligned. 

In [ ]:
import time

import numpy as np
import pandas as pd
import SimpleITK as sitk


LIDC_DIR = DATA_DIR / "TCIA_LIDC-IDRI_20200921"
NODULE_TABLE = REPO_ROOT / "notebooks" / "nodule_table.csv"


def read_series(patient_id, series_uid):
    """Reads one DICOM series as a SimpleITK image in Hounsfield units.

    The series is found by its UID inside the patient folder (searched recursively), not by the folder name
    """
    files = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(str(LIDC_DIR / "lidc_idri" / patient_id), series_uid,
                                                          False, True)   # useSeriesDetails=False, recursive=True
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames(files)
    return reader.Execute()


def cut_series(image, nodules, out_dir):
    #Cuts and saves the patch of every nodule of one series. Returns one row per nodul
    rows = []
    for nodule in nodules.itertuples():
        # pylidc gives (row, column, slice); SimpleITK indices are (column, row, slice)
        centre = np.array(image.TransformContinuousIndexToPhysicalPoint(
            (float(nodule.vox_col), float(nodule.vox_row), float(nodule.vox_slice))))
        header_point = np.array([nodule.x_mm, nodule.y_mm, nodule.z_mm], dtype=float)
        row = {"patch_id": nodule.nodule_uid, "split": "lidc", "scan_id": nodule.scan_id,
               "x_mm": centre[0], "y_mm": centre[1], "z_mm": centre[2],
               "coord_check_mm": float(np.linalg.norm(centre - header_point)),   # NaN when the scan is tilted
               "status": "ok", "outside_fraction": np.nan, "file": ""}
        if not centre_is_inside(image, centre):
            row["status"] = "centre_outside_volume"
        else:
            patch, row["outside_fraction"] = extract_patch(image, centre)
            row["file"] = f"lidc/{nodule.nodule_uid}.npy"
            np.save(out_dir / row["file"], patch)
        rows.append(row)
    return rows


def cut_lidc_patches(out_dir=PATCHES_DIR, limit_series=None):
    #One series at a time, with one manifest per series so that a long run can be resumed 
    table = pd.read_csv(NODULE_TABLE)
    lidc = table[table["dataset"] == "LIDC"]
    (out_dir / "lidc").mkdir(parents=True, exist_ok=True)
    (out_dir / "manifests").mkdir(exist_ok=True)

    series = sorted(lidc["scan_id"].unique())[:limit_series]
    started = time.time()
    for number, series_uid in enumerate(series, start=1):
        manifest_file = out_dir / "manifests" / f"lidc_{series_uid}.csv"
        if manifest_file.exists():
            continue
        nodules = lidc[lidc["scan_id"] == series_uid]
        image = read_series(nodules["patient_id"].iloc[0], series_uid)
        rows = cut_series(image, nodules, out_dir)
        pd.DataFrame(rows).to_csv(manifest_file, index=False)
        if number % 50 == 0:
            print(f"{number}/{len(series)} series, {time.time() - started:.0f} s", flush=True)

    manifest = pd.concat([pd.read_csv(f) for f in sorted((out_dir / "manifests").glob("lidc_*.csv"))], ignore_index=True)
    # Only nodules whose position two independent computations agree on (within 1 mm) are used for training.
    # In a handful of series the slice order differs between pylidc and SimpleITK, and in tilted series the
    # header point does not exist, so their position cannot be confirmed.
    manifest["position_verified"] = manifest["coord_check_mm"] <= 1.0
    manifest.to_csv(out_dir / "manifest_lidc.csv", index=False)
    print(f"{len(manifest)} LIDC nodules, status: {manifest['status'].value_counts().to_dict()}")
    print(f"position verified (within 1 mm of the header point): {int(manifest['position_verified'].sum())}")
    return manifest

## 2.2 Segmentation and features

### Segmenting the nodule and measuring it

The input is a 64 x 64 x 64 patch with 1 mm voxels, so distances are in mm and
volumes in mm3. The only thing we know about the nodule is that its centre is at voxel (32, 32, 32).

Segmentation in five steps:
 1. Smooth the patch lightly (sigma 0.7 mm) to reduce noise.
 2. Threshold halfway between the HU at the centre and the HU of the surrounding lung, which adapts to solid nodules (about 0 HU) and to ground glass ones (about -500 HU).
 3. Keep the connected region that touches the centre.
 4. Vessels and the pleura are also above the threshold and touch the nodule. We measure the widest sphere that fits inside the region near the centre (the nodule itself) and cut everything that lies further than 1.2 times that radius from it.
 5. Keep the connected region that touches the centre again.

Two safeguards were added:
 * Intensities are clipped to [-1024, 3071] HU before any feature is computed. This is the range of a 12 bit CT. Values outside it are padding or artefacts (we found -8192 and 22544 in some patches) and would distort means and percentiles. The IBSI calls this step re-segmentation by intensity range (Zwanenburg et al., 2020).
 * Sphere features. Some findings cannot be segmented (8% of the rated training nodules), and the model then had nothing but imputed values for them. Statistics in spheres of 3, 5 and 8 mm around the centre need no mask at all, so they exist for every finding, in training and in test alike.

In [ ]:
import time
from multiprocessing import Pool

import numpy as np
import pandas as pd
from scipy import ndimage as ndi
from skimage.measure import marching_cubes, mesh_surface_area


FEATURES_FILE = PATCHES_DIR / "features.csv"
SMOOTHING_SIGMA_MM = 0.7
CENTRE_RADIUS_MM = 2          # HU at the centre is the median inside this radius
LUNG_SHELL_MM = (24, 31)      # distances from the centre where we measure the surrounding lung
MAX_EXTENT_MM = 28            # nothing farther than this from the centre can belong to the nodule
INSCRIBED_SEARCH_MM = 1.5     # the widest sphere is searched this close to the centre
GROWTH = 1.2                  # the region is cut at GROWTH x (radius of the widest sphere) + PADDING_MM
PADDING_MM = 1.0
RING_WIDTH_MM = 3
SOLID_HU = -300               # above this we call the tissue solid, below (down to -750) ground glass
GROUND_GLASS_LOW_HU = -750
HU_RANGE = (-1024, 3071)      # range of a 12 bit CT; anything outside is padding or artefact
SPHERE_RADII_MM = (3, 5, 8)   # sphere features: the smallest LNDb nodules, a typical one, a large one

_z, _y, _x = np.indices((64, 64, 64))
DISTANCE_TO_CENTRE = np.sqrt((_z - CENTRE_INDEX) ** 2 + (_y - CENTRE_INDEX) ** 2 + (_x - CENTRE_INDEX) ** 2)


def segment_from_centre(patch):
    """Returns (mask, info). The mask is empty when there is no nodule like structure at the centre."""
    hu = np.clip(patch.astype(np.float32), -1024, 400)
    smooth = ndi.gaussian_filter(hu, SMOOTHING_SIGMA_MM)
    centre_hu = float(np.median(smooth[DISTANCE_TO_CENTRE <= CENTRE_RADIUS_MM]))
    shell = (DISTANCE_TO_CENTRE >= LUNG_SHELL_MM[0]) & (DISTANCE_TO_CENTRE <= LUNG_SHELL_MM[1]) & (smooth < -500)
    lung_hu = float(np.median(smooth[shell])) if shell.sum() > 200 else -850.0
    threshold = min((centre_hu + lung_hu) / 2, SOLID_HU)
    info = {"centre_hu": centre_hu, "lung_hu": lung_hu, "threshold_hu": threshold}
    empty = np.zeros(patch.shape, dtype=bool)
    if centre_hu < lung_hu + 150:                              # the centre is not denser than the lung
        return empty, info

    region = touching_centre(smooth > threshold, DISTANCE_TO_CENTRE <= MAX_EXTENT_MM)
    if region is None:
        return empty, info
    inside_distance = ndi.distance_transform_edt(region)
    near_centre = np.where(DISTANCE_TO_CENTRE <= INSCRIBED_SEARCH_MM, inside_distance, 0)
    widest = np.unravel_index(near_centre.argmax(), region.shape)
    radius = inside_distance[widest]
    distance_to_widest = np.sqrt((_z - widest[0]) ** 2 + (_y - widest[1]) ** 2 + (_x - widest[2]) ** 2)
    cut = region & (distance_to_widest <= GROWTH * radius + PADDING_MM)
    labels, _ = ndi.label(cut)
    return (labels == labels[widest]) if labels[widest] > 0 else cut, info


def touching_centre(mask, allowed):
    """The connected region of mask & allowed that touches the centre, or None."""
    labels, _ = ndi.label(mask & allowed)
    near = labels[DISTANCE_TO_CENTRE <= 3]
    near = near[near > 0]
    return None if len(near) == 0 else labels == np.bincount(near).argmax()


def sphere_features(clipped):
    """HU statistics in spheres centred on the finding. They need no mask, so every finding has them."""
    features = {}
    for radius in SPHERE_RADII_MM:
        values = clipped[DISTANCE_TO_CENTRE <= radius]
        features[f"sphere{radius}_mean_hu"] = float(values.mean())
        features[f"sphere{radius}_p90_hu"] = float(np.percentile(values, 90))
        features[f"sphere{radius}_solid_fraction"] = float((values > SOLID_HU).mean())
    return features


def measure(patch):
    """Segments the patch and returns a dict with the features of the nodule."""
    mask, info = segment_from_centre(patch)                  # the segmentation clips its own copy to [-1024, 400]
    clipped = np.clip(patch.astype(np.float32), *HU_RANGE)   # every intensity feature uses the clipped patch
    features = {**info, "segmented": bool(mask.any()), **sphere_features(clipped)}
    names = ["volume_mm3", "diameter_mm", "max_extent_mm", "elongation", "flatness", "sphericity",
             "hu_mean", "hu_std", "hu_p10", "hu_p50", "hu_p90", "hu_skew", "solid_fraction", "ground_glass_fraction",
             "edge_gradient", "ring_mean_hu", "ring_solid_fraction", "contrast_hu"]
    if not mask.any() or mask.sum() < 8:
        return {**features, **dict.fromkeys(names, np.nan)}

    points = np.argwhere(mask).astype(float)
    volume = float(mask.sum())
    axes = np.sqrt(np.maximum(np.linalg.eigvalsh(np.cov(points.T)), 1e-9))[::-1]       # large to small
    area = surface_area(mask)
    hu = clipped[mask].astype(float)
    ring = ndi.binary_dilation(mask, iterations=RING_WIDTH_MM) & ~ndi.binary_dilation(mask, iterations=1)
    ring_hu = clipped[ring].astype(float)
    edge = mask & ~ndi.binary_erosion(mask)
    gradient = ndi.gaussian_gradient_magnitude(clipped, SMOOTHING_SIGMA_MM)
    return {**features,
            "volume_mm3": volume,
            "diameter_mm": 2 * (3 * volume / (4 * np.pi)) ** (1 / 3),
            "max_extent_mm": float(np.ptp(points @ axis_of_largest_spread(points))) + 1,
            "elongation": axes[1] / axes[0], "flatness": axes[2] / axes[0],
            "sphericity": np.pi ** (1 / 3) * (6 * volume) ** (2 / 3) / area,
            "hu_mean": hu.mean(), "hu_std": hu.std(), "hu_p10": np.percentile(hu, 10),
            "hu_p50": np.percentile(hu, 50), "hu_p90": np.percentile(hu, 90),
            "hu_skew": float(((hu - hu.mean()) ** 3).mean() / (hu.std() ** 3 + 1e-9)),
            "solid_fraction": (hu > SOLID_HU).mean(),
            "ground_glass_fraction": ((hu > GROUND_GLASS_LOW_HU) & (hu <= SOLID_HU)).mean(),
            "edge_gradient": float(gradient[edge].mean()),
            "ring_mean_hu": ring_hu.mean() if len(ring_hu) else np.nan,
            "ring_solid_fraction": (ring_hu > SOLID_HU).mean() if len(ring_hu) else np.nan,
            "contrast_hu": hu.mean() - (ring_hu.mean() if len(ring_hu) else np.nan)}


def axis_of_largest_spread(points):
    """Unit vector of the direction in which the points are most spread out."""
    return np.linalg.eigh(np.cov(points.T))[1][:, -1]


def surface_area(mask):
    """Area in mm2 of the surface of the mask (marching cubes on the padded mask)."""
    vertices, faces, _, _ = marching_cubes(np.pad(mask, 1).astype(np.float32), level=0.5)
    return float(mesh_surface_area(vertices, faces))


def measure_file(path):
    try:
        return measure(np.load(PATCHES_DIR / path))
    except Exception as error:
        return {"segmented": False, "error": str(error)}


def measure_all_patches():
    # LNDb train and test, plus the LIDC patches when they have been cut (see lidc_patches.py)
    splits = [s for s in ("train", "test", "lidc") if (PATCHES_DIR / f"manifest_{s}.csv").exists()]
    manifest = pd.concat([pd.read_csv(PATCHES_DIR / f"manifest_{s}.csv") for s in splits], ignore_index=True)
    ok = manifest[manifest["status"] == "ok"]
    started = time.time()
    with Pool(6) as pool:
        rows = pool.map(measure_file, ok["file"].tolist(), chunksize=20)
    features = pd.concat([ok[["patch_id", "split"]].reset_index(drop=True), pd.DataFrame(rows)], axis=1)
    features.to_csv(FEATURES_FILE, index=False)
    print(f"{len(features)} patches measured in {time.time() - started:.0f} s, saved {FEATURES_FILE}")
    print("segmented:", features.groupby("split")["segmented"].mean().round(3).to_dict())

### PyRadiomics with an IBSI oriented configuration, and ICC

PyRadiomics is a library which helps calculate hundreds of features of the images inside a desired region.


Configuration, and why (IBSI: Zwanenburg et al., 2020; PyRadiomics: van Griethuysen et al., 2017):

 * Isotropic 1 mm voxels with linear interpolation: already done when the patch is cut so PyRadiomics does not resample again. Texture features depend on voxe size (Shafiq-ul-Hassan et al., 2017).
 * Intensity range [-1024, 3071] HU before discretisation (IBSI re-segmentation by range).
 * Fixed bin width of 25 HU. For calibrated units such as HU the IBSI recommends a fixed bin size, so that a bin means the same density in every scan; 25 HU is the PyRadiomics default and widely used in CT.
 * Feature classes: first order, shape, GLCM, GLRLM, GLSZM, GLDM and NGTDM, on the original image and on Laplacian of Gaussian filtered images with sigma 1, 2 and 3 mm (edges and blobs at the scales of small nodules). PyRadiomics documents where its definitions deviate from the IBSI.
 * Region of interest: the same automatic mask in training and test (nodule_features.segment_from_centre), because the test file only has centres (Pedrosa et al., 2019). The radiologists' masks are used only to measure robustness.

Robustness. What the ICC measures?

Two radiologists never draw exactly the same contour, so a feature computed from each of their masks will differ a little on the same nodule. The intraclass correlation coefficient (ICC) asks whether that difference is small compared with the differences between nodules: it is the share of the total variability that comes from real differences between nodules rather than from who drew the contour. An ICC of 1 means the two radiologists give the same value for every nodule; an ICC of 0 means the disagreement between them is as large as the differences between nodules, so the feature describes the contour more than the nodule. 

We use ICC(2,1) (Shrout and Fleiss, 1979): absolute agreement, so that a systematic difference (one radiologist always drawing larger contours) also lowers the score, unlike a plain correlation; random raters, because the feature should hold for any radiologist and not only for these five; and a single rater, because the model uses one mask per nodule. Following Koo and Li (2016), we call a feature robust when its ICC is at least 0.75.

In [ ]:
import logging
from multiprocessing import Pool

import numpy as np
import pandas as pd
import SimpleITK as sitk
from radiomics import featureextractor


SETTINGS = {"binWidth": 25, "resampledPixelSpacing": None, "interpolator": "sitkLinear", "label": 1,
            "minimumROIDimensions": 1, "minimumROISize": 8, "geometryTolerance": 1e-3}
LOG_SIGMAS_MM = [1.0, 2.0, 3.0]
ROBUST_ICC = 0.75
logging.getLogger("radiomics").setLevel(logging.ERROR)


def make_extractor():
    extractor = featureextractor.RadiomicsFeatureExtractor(**SETTINGS)
    extractor.enableAllFeatures()
    extractor.enableImageTypes(Original={}, LoG={"sigma": LOG_SIGMAS_MM})
    return extractor


_EXTRACTOR = None


def extract(patch, mask):
    """Radiomic features of one region. Returns an empty dict when the region is too small."""
    global _EXTRACTOR
    _EXTRACTOR = _EXTRACTOR or make_extractor()        # one extractor per worker process
    if mask.sum() < SETTINGS["minimumROISize"]:
        return {}
    image = sitk.GetImageFromArray(np.clip(patch.astype(np.float32), *HU_RANGE))   # spacing 1 mm by construction
    region = sitk.GetImageFromArray(mask.astype(np.uint8))
    values = _EXTRACTOR.execute(image, region)
    return {name: float(value) for name, value in values.items() if not name.startswith("diagnostics")}


def icc_2_1(ratings):
    """ICC(2,1) of an (n subjects x k raters) array: two way random effects, absolute agreement, single rater."""
    n, k = ratings.shape
    grand = ratings.mean()
    ms_rows = k * ((ratings.mean(axis=1) - grand) ** 2).sum() / (n - 1)
    ms_cols = n * ((ratings.mean(axis=0) - grand) ** 2).sum() / (k - 1)
    residual = ratings - ratings.mean(axis=1, keepdims=True) - ratings.mean(axis=0, keepdims=True) + grand
    ms_error = (residual ** 2).sum() / ((n - 1) * (k - 1))
    return (ms_rows - ms_error) / (ms_rows + (k - 1) * ms_error + k * (ms_cols - ms_error) / n)


def _work(item):
    patch_id, roi, patch, mask = item
    try:
        return {"patch_id": patch_id, "roi": roi, **extract(patch, mask)}
    except Exception as error:                # one failing region must not stop the run
        return {"patch_id": patch_id, "roi": roi, "error": str(error)}


def training_items():
    """Automatic mask and every radiologist mask of each rated LNDb training nodule."""
    table = pd.read_csv(REPO_ROOT / "notebooks" / "nodule_table.csv")
    rated = table[(table["dataset"] == "LNDb") & table["mal_mean"].notna()]
    merged = pd.read_csv(LNDB_DIR / "LNDb_csvs" / "trainNodules_gt.csv")
    merged["nodule_uid"] = [f"LNDb-{c:04d}_{f}" for c, f in zip(merged["LNDbID"], merged["FindingID"])]
    merged = merged[merged["nodule_uid"].isin(rated["nodule_uid"])]
    for ct_id, findings in merged.groupby("LNDbID"):
        masks = {}
        for f in findings.itertuples():
            patch_id = f"LNDb-{ct_id:04d}_f{f.FindingID:03d}"
            patch = np.load(PATCHES_DIR / "train" / f"{patch_id}.npy")
            yield patch_id, "auto", patch, segment_from_centre(patch)[0]
            for rad, finding in zip(str(f.RadID).split(","), str(f.RadFindingID).split(",")):
                path = LNDB_DIR / "LNDb_masks" / "masks" / f"LNDb-{ct_id:04d}_rad{rad}.mhd"
                if not path.exists():
                    continue
                masks.setdefault(rad, sitk.ReadImage(str(path)))
                mask_patch, _ = extract_patch(masks[rad], (f.x, f.y, f.z), interpolator=sitk.sitkNearestNeighbor, outside_value=0)
                yield patch_id, f"rad{rad}", patch, mask_patch == int(finding)


def robustness_table(features):
    """ICC(2,1) of every feature between the first two radiologist masks of the same nodule."""
    readers = features[features["roi"].str.startswith("rad")].dropna(axis=1, how="all")
    names = [c for c in readers.columns if c.startswith(("original_", "log-"))]
    readers = readers.dropna(subset=names)    # masks under 8 voxels (findings < 3 mm have only a centre) give no features
    pairs =readers.groupby("patch_id").filter(lambda g: len(g) >= 2).groupby("patch_id").head(2).sort_values(["patch_id", "roi"])
    first, second = pairs.groupby("patch_id").nth(0)[names].to_numpy(), pairs.groupby("patch_id").nth(1)[names].to_numpy()
    icc = {name: icc_2_1(np.column_stack([first[:, i], second[:, i]])) for i, name in enumerate(names)}
    table = pd.DataFrame({"feature": list(icc), "icc": list(icc.values())})
    table["image"] = table["feature"].str.split("_").str[0]
    table["feature_class"] = table["feature"].str.split("_").str[1]
    table["robust"] = table["icc"] >= ROBUST_ICC
    return table, len(first)


def extract_training_radiomics(recompute=True):
    if recompute:
        with Pool(4) as pool:
            rows = pool.map(_work, list(training_items()), chunksize=8)
        pd.DataFrame(rows).to_csv(PATCHES_DIR / "radiomics_train.csv", index=False)
    features = pd.read_csv(PATCHES_DIR / "radiomics_train.csv")
    table, n_nodules = robustness_table(features)
    table.to_csv(PATCHES_DIR / "radiomics_icc.csv", index=False)
    print(f"{len(features)} regions; ICC from {n_nodules} nodules with two masks; robust: {table['robust'].mean():.0%}")

## 2.3 Model

### Ordinal logistic regression

#### Why this model?

The malignancy ratings 1 to 5 are ordered, and the competition scores errors by their distance (QWK and MAE). Moreover, taking into consideration the CRISP-DM Methodology we followed, the Business Understanding phase also highlights the difference between mistaking a 1 for a 2 or a 1 for a 5 in a clinical environment. The proportional odds model (McCullagh, 1980) respects that order with very few parameters: one weight per feature, shared by all classes, and four cut points on a single latent score:

    P(rating <= k) = sigmoid(cut_k - x . w),   k = 1, 2, 3, 4,   with cut_1 < cut_2 < cut_3 < cut_4

so P(rating = k) = P(rating <= k) - P(rating <= k - 1). A multinomial logistic regression would need four weights per feature, which is a lot for 765 labelled nodules, and a regression rounded into classes ignores that the gaps between classes need not be equal. Gutierrez et al. (2016) compare ordinal methods and find this simple threshold model competitive.

##### Regularisation

We add an L2 penalty alpha/2 * |w|^2 on the weights (not on the cut points), as in ridge logistic regression (le Cessie and van Houwelingen, 1992). It keeps the weights stable when features are correlated, which is the case for size and intensity features.

##### Decision rule
predict() returns the most probable class (argmax). The best point prediction depends on the metric: the most probable class is optimal for accuracy, and the median of the predicted distribution is optimal for the absolute error (Gneiting, 2011). In our validation both gave a much lower MAE than cutting
a regression score, so we keep argmax, and predict_median() is available to compare.

The cut points are kept in order by writing them as the first cut plus positive steps (exp of a free value).

In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.base import BaseEstimator, ClassifierMixin


class OrdinalLogisticRegression(BaseEstimator, ClassifierMixin):
    """Proportional odds model with an L2 penalty. Use it after a StandardScaler, like any penalised model."""

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def _unpack(self, params, n_features):
        weights = params[:n_features]
        cuts = np.cumsum(np.r_[params[n_features], np.exp(params[n_features + 1:])])   # always increasing
        return weights, cuts

    def _cumulative(self, X, weights, cuts):
        """P(rating <= k) for every row and every class, with 0 before the first class and 1 after the last."""
        below = expit(cuts[None, :] - (X @ weights)[:, None])
        return np.hstack([np.zeros((len(X), 1)), below, np.ones((len(X), 1))])

    def fit(self, X, y, sample_weight=None):
        X, y = np.asarray(X, dtype=float), np.asarray(y)
        self.classes_ = np.unique(y)
        index = np.searchsorted(self.classes_, y)            # class position 0..K-1
        weight = np.ones(len(y)) if sample_weight is None else np.asarray(sample_weight, dtype=float)
        n_features, rows = X.shape[1], np.arange(len(y))

        n_cuts = len(self.classes_) - 1

        def loss_and_gradient(params):
            """Penalised negative log likelihood and its exact gradient (the optimiser converges reliably with it)."""
            weights, cuts = self._unpack(params, n_features)
            cumulative = self._cumulative(X, weights, cuts)
            likelihood = np.clip(cumulative[rows, index + 1] - cumulative[rows, index], 1e-12, None)
            loss = -(weight * np.log(likelihood)).sum() + 0.5 * self.alpha * weights @ weights

            # The derivative of sigmoid(z) is s(1 - s). Upper and lower are the two cut points around each rating.
            density = cumulative[:, 1:-1] * (1 - cumulative[:, 1:-1])
            density = np.hstack([np.zeros((len(X), 1)), density, np.zeros((len(X), 1))])
            upper, lower = density[rows, index + 1] / likelihood, density[rows, index] / likelihood
            grad_weights = X.T @ (weight * (upper - lower)) + self.alpha * weights
            grad_cuts = (np.bincount(index, -weight * upper, minlength=n_cuts + 1)[:n_cuts]
                         + np.bincount(index, weight * lower, minlength=n_cuts + 1)[1:])
            # Chain rule for cuts = first + cumulative sum of exp(steps)
            steps = np.exp(params[n_features + 1:])
            grad_steps = steps * np.cumsum(grad_cuts[::-1])[::-1][1:]
            return loss, np.r_[grad_weights, grad_cuts.sum(), grad_steps]

        # Start: all weights at zero and cut points that reproduce the class frequencies of the training data
        share_below = np.cumsum(np.bincount(index, minlength=len(self.classes_)))[:-1] / len(y)
        start_cuts = np.log(share_below / (1 - share_below))
        start = np.r_[np.zeros(n_features), start_cuts[0], np.log(np.maximum(np.diff(start_cuts), 1e-3))]
        result = minimize(loss_and_gradient, start, jac=True, method="L-BFGS-B", options={"maxiter": 2000})
        self.coef_, self.cuts_ = self._unpack(result.x, n_features)
        self.converged_ = bool(result.success)
        return self

    def predict_proba(self, X):
        cumulative = self._cumulative(np.asarray(X, dtype=float), self.coef_, self.cuts_)
        return np.diff(cumulative, axis=1)

    def predict(self, X):
        """Most probable class (argmax)."""
        return self.classes_[self.predict_proba(X).argmax(axis=1)]

    def predict_median(self, X):
        """Median of the predicted distribution: the first class whose cumulative probability reaches 0.5."""
        return self.classes_[(self.predict_proba(X).cumsum(axis=1) >= 0.5).argmax(axis=1)]